Matching:
ASD:
EF-ToM
1170-3200
1161-3081
1121-3041
1101-3061
1141-3161
1151-3071
1191-3111
1031-3121
1021-3021
1011-3011
1071-3051

TD:
2360-4150
2121-4181
2201-4171
2250-4210
2281-4091
2340-4080
2151-4071
2211-4141

In [5]:
import pandas as pd
import numpy as np
import mne
from pathlib import Path
import os

output_dir = r'H:\ASD projoect\EF_ToM'
pairs = [
    # ASD
    ("ASD", "1170", "3200"),
    ("ASD", "1161", "3081"),
    ("ASD", "1121", "3041"),
    ("ASD", "1101", "3061"),
    ("ASD", "1141", "3161"),
    ("ASD", "1151", "3071"),
    ("ASD", "1191", "3111"),
    ("ASD", "1031", "3121"),
    ("ASD", "1021", "3021"),
    ("ASD", "1011", "3011"),
    ("ASD", "1071", "3051"),

    # TD
    ("TD", "2360", "4150"),
    ("TD", "2121", "4181"),
    ("TD", "2201", "4171"),
    ("TD", "2250", "4210"),
    ("TD", "2281", "4091"),
    ("TD", "2340", "4080"),
    ("TD", "2151", "4071"),
    ("TD", "2211", "4141"),
]

pair_df = pd.DataFrame(
    pairs,
    columns=["group", "EF_ID", "ToM_ID"]
)

pair_df

,group,EF_ID,ToM_ID
0,ASD,1170,3200
1,ASD,1161,3081
2,ASD,1121,3041
3,ASD,1101,3061
4,ASD,1141,3161
5,ASD,1151,3071
6,ASD,1191,3111
7,ASD,1031,3121
8,ASD,1021,3021
9,ASD,1011,3011


In [4]:

def extract_tom_erp(input_file, output_file, segment_name):

    # -----------------------------
    # Load interpolated data
    # -----------------------------
    data = np.load(
        input_file,
        allow_pickle=True
    ).item()

    metadata = pd.DataFrame(
        data["metadata"]
    )

    # -----------------------------
    # Reconstruct epochs
    # -----------------------------
    info = mne.create_info(
        ch_names=list(data["channels"]),
        sfreq=data["sfreq"],
        ch_types="eeg"
    )

    montage = mne.channels.make_standard_montage(
        "standard_1020"
    )

    info.set_montage(
        montage,
        match_case=False,
        on_missing="ignore"
    )

    epochs = mne.EpochsArray(
        data["eeg"],
        info,
        tmin=data["times"][0],
        metadata=metadata,
        verbose=False
    )

    # same reference as EF
    epochs.set_eeg_reference(
        "average",
        projection=False
    )

    epochs.apply_baseline(
        (-0.2, 0)
    )

    # -----------------------------
    # Conditions
    # -----------------------------
    conditions = {
        "false_cog":
            (metadata["belief"] == "false") &
            (metadata["tom"] == "cognitive"),

        "true_cog":
            (metadata["belief"] == "true") &
            (metadata["tom"] == "cognitive"),

        "false_aff":
            (metadata["belief"] == "false") &
            (metadata["tom"] == "affective"),

        "true_aff":
            (metadata["belief"] == "true") &
            (metadata["tom"] == "affective")
    }

    # -----------------------------
    # ERP definitions
    # -----------------------------
    posterior = [
        "CPZ", "CP1", "CP2", "CP3", "CP4",
        "PZ", "P1", "P2", "P3", "P4",
        "POZ", "PO3", "PO4"
    ]

    erp_defs = {
        "LPC": {
            "channels": [
                "FZ", "F1", "F2",
                "FCZ", "FC1", "FC2"
            ],
            "tmin": 0.300,
            "tmax": 0.600
        },

        "early_LSW": {
            "channels": posterior,
            "tmin": 0.800,
            "tmax": 1.400
        },

        "mid_LSW": {
            "channels": posterior,
            "tmin": 1.400,
            "tmax": 2.100
        },

        "late_LSW": {
            "channels": posterior,
            "tmin": 2.000,
            "tmax": 2.500
        }
    }

    results = {
        "segment": segment_name
    }

    # -----------------------------
    # ERP amplitudes
    # -----------------------------
    for component, cfg in erp_defs.items():

        use_chs = [
            ch for ch in cfg["channels"]
            if ch in epochs.ch_names
        ]

        for condition, mask in conditions.items():

            idx = np.where(
                mask.to_numpy()
            )[0]

            ep = (
                epochs[idx]
                .copy()
                .pick(use_chs)
                .crop(
                    cfg["tmin"],
                    cfg["tmax"]
                )
            )

            x = ep.get_data() * 1e6

            results[
                f"{component}_{condition}"
            ] = x.mean()

            results[
                f"n_{condition}"
            ] = len(idx)

    # -----------------------------
    # False - True contrasts
    # -----------------------------
    for component in erp_defs:

        results[
            f"delta_{component}_cog"
        ] = (
            results[f"{component}_false_cog"]
            -
            results[f"{component}_true_cog"]
        )

        results[
            f"delta_{component}_aff"
        ] = (
            results[f"{component}_false_aff"]
            -
            results[f"{component}_true_aff"]
        )

    # -----------------------------
    # Save
    # -----------------------------
    np.save(
        output_file,
        results,
        allow_pickle=True
    )

    print(
        segment_name,
        "saved:",
        output_file
    )

    return results

In [7]:
from pathlib import Path

ROOTS = {
    "ASD": {
        "EF": Path(r"H:\ASD projoect\EF\ASD"),
        "ToM": Path(r"H:\ASD projoect\ToM\ASD"),
    },

    "TD": {
        "EF": Path(r"H:\ASD projoect\EF\TD"),
        "ToM": Path(r"H:\ASD projoect\ToM\TD"),
    }
}

In [8]:
for group, ef_id, tom_id in pairs:

    tom_dir = (
        ROOTS[group]["ToM"]
        / f"{tom_id}a_CV_processed_numpy"
    )

    print("\nProcessing:", group, tom_id)

    # --------------------------------
    # Hearing
    # --------------------------------
    hearing_file = (
        tom_dir
        / "interpolated_hearing_epoch.npy"
    )

    if hearing_file.exists():

        extract_tom_erp(
            hearing_file,
            tom_dir / "ToM_ERP_hearing_results.npy",
            segment_name="hearing"
        )

    else:
        print("Missing hearing:", hearing_file)

    # --------------------------------
    # Thinking
    # --------------------------------
    thinking_file = (
        tom_dir
        / "interpolated_thinking_epoch.npy"
    )

    if thinking_file.exists():

        extract_tom_erp(
            thinking_file,
            tom_dir / "ToM_ERP_thinking_results.npy",
            segment_name="thinking"
        )

    else:
        print("Missing thinking:", thinking_file)


Processing: ASD 3200
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Applying baseline correction (mode: mean)
hearing saved: H:\ASD projoect\ToM\ASD\3200a_CV_processed_numpy\ToM_ERP_hearing_results.npy
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Applying baseline correction (mode: mean)
thinking saved: H:\ASD projoect\ToM\ASD\3200a_CV_processed_numpy\ToM_ERP_thinking_results.npy

Processing: ASD 3081
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Applying baseline correction (mode: mean)
hearing saved: H:\ASD projoect\ToM\ASD\3081a_CV_processed_numpy\ToM_ERP_hearing_results.npy
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Applying baseline correction (mode: mean)
thinking saved: H:\ASD projoect\ToM\ASD\3081a_CV_processed_numpy\T

In [9]:
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

def flatten_dict(d, prefix=""):

    out = {}

    for k, v in d.items():

        name = f"{prefix}_{k}" if prefix else str(k)

        if isinstance(v, dict):
            out.update(flatten_dict(v, name))

        elif np.isscalar(v):
            out[name] = v

    return out

In [13]:
def load_group_results(group, pair_list):

    rows = []

    for ef_id, tom_id in pair_list:

        ef_file = (
            ROOTS[group]["EF"]
            / f"{ef_id}_CF_CV_processed_numpy"
            / "EF_ERP_results.npy"
        )

        tom_dir = (
            ROOTS[group]["ToM"]
            / f"{tom_id}a_CV_processed_numpy"
        )

        hearing_file = (
            tom_dir / "ToM_ERP_hearing_results.npy"
        )

        thinking_file = (
            tom_dir / "ToM_ERP_thinking_results.npy"
        )

        ef = flatten_dict(
            np.load(
                ef_file,
                allow_pickle=True
            ).item()
        )

        hearing = flatten_dict(
            np.load(
                hearing_file,
                allow_pickle=True
            ).item()
        )

        thinking = flatten_dict(
            np.load(
                thinking_file,
                allow_pickle=True
            ).item()
        )

        row = {
            "EF_ID": ef_id,
            "ToM_ID": tom_id
        }

        row.update({
            f"EF_{k}": v
            for k, v in ef.items()
        })

        row.update({
            f"H_{k}": v
            for k, v in hearing.items()
        })

        row.update({
            f"T_{k}": v
            for k, v in thinking.items()
        })

        rows.append(row)

    return pd.DataFrame(rows)

In [12]:
ASD_pairs = [
    ("1170", "3200"),
    ("1161", "3081"),
    ("1121", "3041"),
    ("1101", "3061"),
    ("1141", "3161"),
    ("1151", "3071"),
    ("1191", "3111"),
    ("1031", "3121"),
    ("1021", "3021"),
    ("1011", "3011"),
    ("1071", "3051"),
]

TD_pairs = [
    ("2360", "4150"),
    ("2121", "4181"),
    ("2201", "4171"),
    ("2250", "4210"),
    ("2281", "4091"),
    ("2340", "4080"),
    ("2151", "4071"),
    ("2211", "4141"),
]

In [16]:
ASD = load_group_results(
    "ASD",
    ASD_pairs
)

TD = load_group_results(
    "TD",
    TD_pairs
)

print("ASD:", ASD.shape)
print("TD:", TD.shape)

ASD: (11, 72)
TD: (8, 72)


In [18]:
print(ASD)

   EF_ID ToM_ID  EF_N2_go_amplitude_uv  EF_N2_nogo_amplitude_uv  \
0   1170   3200             -13.911164               -15.998488   
1   1161   3081              -9.202507               -14.539873   
2   1121   3041              -1.329541                -1.307636   
3   1101   3061             -14.684662               -14.968939   
4   1141   3161              -8.050844                -8.279713   
5   1151   3071              -4.550767                -5.000819   
6   1191   3111               5.904471                 1.464682   
7   1031   3121              -7.139818                -5.544189   
8   1021   3021              -5.189782                -8.759604   
9   1011   3011              -2.751808                -6.288115   
10  1071   3051              -8.900363               -14.507561   

    EF_N2_nogo_go_effect_uv  EF_N2_go_latency_ms  EF_N2_nogo_latency_ms  \
0                 -2.087324                344.0                  341.0   
1                 -5.337366                35

In [19]:
EF_features = [
    c for c in ASD.columns
    if c.startswith("EF_")
    and "effect" in c.lower()
]

print(EF_features)

['EF_N2_nogo_go_effect_uv', 'EF_P3_LPP_nogo_go_effect_uv']


In [20]:
hearing_features = [
    c for c in ASD.columns
    if c.startswith("H_delta_")
]

thinking_features = [
    c for c in ASD.columns
    if c.startswith("T_delta_")
]

print("EF:")
print(EF_features)

print("\nHearing:")
print(hearing_features)

print("\nThinking:")
print(thinking_features)

EF:
['EF_N2_nogo_go_effect_uv', 'EF_P3_LPP_nogo_go_effect_uv']

Hearing:
['H_delta_LPC_cog', 'H_delta_LPC_aff', 'H_delta_early_LSW_cog', 'H_delta_early_LSW_aff', 'H_delta_mid_LSW_cog', 'H_delta_mid_LSW_aff', 'H_delta_late_LSW_cog', 'H_delta_late_LSW_aff']

Thinking:
['T_delta_LPC_cog', 'T_delta_LPC_aff', 'T_delta_early_LSW_cog', 'T_delta_early_LSW_aff', 'T_delta_mid_LSW_cog', 'T_delta_mid_LSW_aff', 'T_delta_late_LSW_cog', 'T_delta_late_LSW_aff']


In [21]:
for df in [ASD, TD]:

    df["EF_N2_latency_effect_ms"] = (
        df["EF_N2_nogo_latency_ms"]
        - df["EF_N2_go_latency_ms"]
    )

    df["EF_P3_LPP_latency_effect_ms"] = (
        df["EF_P3_LPP_nogo_latency_ms"]
        - df["EF_P3_LPP_go_latency_ms"]
    )

In [22]:
def correlation_table(df, ef_features, tom_features):

    results = []

    for ef in ef_features:

        for tom in tom_features:

            x = pd.to_numeric(
                df[ef],
                errors="coerce"
            )

            y = pd.to_numeric(
                df[tom],
                errors="coerce"
            )

            valid = x.notna() & y.notna()

            rho, p = spearmanr(
                x[valid],
                y[valid]
            )

            results.append({
                "EF_feature": ef,
                "ToM_feature": tom,
                "n": valid.sum(),
                "rho": rho,
                "p": p
            })

    results = pd.DataFrame(results)

    # FDR correction
    results["p_FDR"] = multipletests(
        results["p"],
        method="fdr_bh"
    )[1]

    return results.sort_values(
        "p"
    ).reset_index(drop=True)

In [23]:
ASD_hearing_corr = correlation_table(
    ASD,
    EF_features,
    hearing_features
)

ASD_thinking_corr = correlation_table(
    ASD,
    EF_features,
    thinking_features
)

TD_hearing_corr = correlation_table(
    TD,
    EF_features,
    hearing_features
)

TD_thinking_corr = correlation_table(
    TD,
    EF_features,
    thinking_features
)

In [24]:
# save results at H:\ASD projoect\EF_ToM
results_path = r'H:\ASD projoect\EF_ToM'
results_name = "EF_ToM_correlation_results.xlsx"
results_file = os.path.join(results_path, results_name)

with pd.ExcelWriter(results_file) as writer:
    ASD_hearing_corr.to_excel(
        writer,
        sheet_name="ASD_hearing",
        index=False
    )

    ASD_thinking_corr.to_excel(
        writer,
        sheet_name="ASD_thinking",
        index=False
    )

    TD_hearing_corr.to_excel(
        writer,
        sheet_name="TD_hearing",
        index=False
    )

    TD_thinking_corr.to_excel(
        writer,
        sheet_name="TD_thinking",
        index=False
    )